整體執行流程  
Step 1: Python 生成 ad_campaigns.csv  
Step 2: 上傳 CSV 到 BigQuery（建立補充表）  
Step 3: GA4 Public Dataset 查詢（建立流量轉換視圖）  
Step 4: BigQuery JOIN 合併兩表，計算 ROI  
Step 5: Power BI 連接 BigQuery，建立 Dashboard  


Step 1：Python 生成 ad_campaigns.csv
在你的本機 Jupyter Notebook 或 VS Code 執行，儲存到 data/ 資料夾：

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

channels = ['google', 'facebook', 'email', 'organic']
campaigns = {
    'google':   ['Google_Brand_Search', 'Google_Shopping', 'Google_Display'],
    'facebook': ['FB_Retargeting', 'FB_Lookalike', 'FB_Awareness'],
    'email':    ['Email_Newsletter', 'Email_Promo', 'Email_Winback'],
    'organic':  ['Organic_SEO', 'Organic_Social']
}

rows = []
for channel, camp_list in campaigns.items():
    for camp in camp_list:
        rows.append({
            'campaign_name': camp,
            'channel':       channel,
            'ad_spend':      round(np.random.uniform(500, 5000), 2),
            'impressions':   np.random.randint(10000, 200000),
            'clicks':        np.random.randint(500, 8000),
        })

df = pd.DataFrame(rows)
df.to_csv('data/ad_campaigns.csv', index=False)
print(df)


Step 2：上傳 CSV 到 BigQuery
在 BigQuery Console：

建立 Dataset，命名例如 traffic_roi

建立 Table → 來源選「Upload CSV」→ 選 ad_campaigns.csv

Table 命名：ad_campaigns，Auto-detect schema

或用 Python google-cloud-bigquery 套件上傳：

In [ ]:
from google.cloud import bigquery
client = bigquery.Client()
table_id = "your_project.traffic_roi.ad_campaigns"
job = client.load_table_from_dataframe(df, table_id)
job.result()


Step 3：GA4 Public Dataset — 建立流量轉換 CTE
在 BigQuery 查詢編輯器，先跑這段建立基礎視圖，儲存為 bigquery/queries/01_ga4_traffic_sessions.sql：

In [ ]:
-- GA4 流量來源 + 轉換彙總
WITH sessions AS (
  SELECT
    trafficSource.medium                          AS channel,
    trafficSource.source                          AS source,
    trafficSource.campaign                        AS campaign_name,
    COUNT(DISTINCT fullVisitorId)                 AS sessions,
    SUM(totals.transactions)                      AS transactions,
    SUM(totals.transactionRevenue) / 1000000      AS revenue
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
    AND trafficSource.medium IS NOT NULL
  GROUP BY 1, 2, 3
)
SELECT * FROM sessions
ORDER BY revenue DESC;


注意：GA4 Public Dataset 實際是 Universal Analytics 格式（ga_sessions_*），欄位用 trafficSource.medium / .campaign。

Step 4：JOIN 兩表，計算 ROI
儲存為 bigquery/queries/02_roi_analysis.sql：

In [ ]:
WITH ga4_traffic AS (
  SELECT
    trafficSource.medium      AS channel,
    trafficSource.campaign    AS campaign_name,
    COUNT(DISTINCT fullVisitorId)            AS sessions,
    SUM(totals.transactions)                 AS conversions,
    SUM(totals.transactionRevenue) / 1000000 AS revenue
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY 1, 2
),
ad_data AS (
  SELECT
    channel,
    campaign_name,
    ad_spend,
    impressions,
    clicks,
    ROUND(clicks / impressions * 100, 2) AS ctr_pct
  FROM `your_project.traffic_roi.ad_campaigns`
)
SELECT
  g.channel,
  g.campaign_name,
  g.sessions,
  g.conversions,
  g.revenue,
  a.ad_spend,
  a.impressions,
  a.clicks,
  a.ctr_pct,
  ROUND((g.revenue - a.ad_spend) / NULLIF(a.ad_spend, 0) * 100, 2) AS roi_pct,
  ROUND(g.revenue / NULLIF(a.ad_spend, 0), 2)                       AS roas
FROM ga4_traffic g
LEFT JOIN ad_data a
  ON LOWER(g.channel) = LOWER(a.channel)
  AND LOWER(g.campaign_name) = LOWER(a.campaign_name)
ORDER BY roi_pct DESC;


Step 5：Power BI 連接 BigQuery
Power BI Desktop → Get Data → Google BigQuery

輸入 Project ID，選擇 traffic_roi dataset

匯入 02_roi_analysis 查詢結果

建議建立以下頁面：

Page 1：各 Channel 流量 vs 轉換率（Bar + Line combo chart）

Page 2：CTR vs Conversion Rate 散點圖（關聯性分析）

Page 3：各 Campaign ROI / ROAS 排行（Horizontal Bar）



| 檔案                          | 位置                |
| --------------------------- | ----------------- |
| generate_ad_campaigns.ipynb | data/             |
| ad_campaigns.csv            | data/             |
| 01_ga4_traffic_sessions.sql | bigquery/queries/ |
| 02_roi_analysis.sql         | bigquery/queries/ |
| ad_campaigns_schema.json    | bigquery/schema/  |